In [4]:
import requests
import json
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="../api/.env")

# ── Configuration ─────────────────────────────────────────────────────────────
BASE_URL = "http://localhost:8082"
BEARER_TOKEN = os.getenv("API_BEARER_TOKEN", "")

HEADERS = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {BEARER_TOKEN}"
}

# ── Test clinical note ─────────────────────────────────────────────────────────
CLINICAL_TEXT = """Swollen LL, limited mobility, pain and redness over R LL. Possible PE post THR and TKR. 
IV Streptokinase stat
IV NS 500ml run fast
SC Clean 200U stat
Refer to IR for possible embolectomy"""

print("Configuration loaded.")
print(f"  Base URL : {BASE_URL}")
print(f"  Token    : {BEARER_TOKEN[:10]}***" if BEARER_TOKEN else "  Token    : NOT SET")

Configuration loaded.
  Base URL : http://localhost:8082
  Token    : c73e3c54-b***


In [5]:
# ── Block 2: Health Check ─────────────────────────────────────────────────────
print("=" * 60)
print("Health Check (GET / and GET /health)")
print("=" * 60)

for path in ["/", "/health"]:
    r = requests.get(f"{BASE_URL}{path}", headers=HEADERS, timeout=10)
    status = "✓" if r.status_code == 200 else "✗"
    print(f"{status} {path}  →  {r.status_code}  {r.json()}")

Health Check (GET / and GET /health)
✓ /  →  200  {'message': 'CTakes REST Service API'}
✓ /health  →  200  {'status': 'healthy'}


In [6]:
# ── Block 3: Generate Note (Step 1 — LLM Summary) ────────────────────────────
print("=" * 60)
print("Generate Note  (POST /generate/note)")
print("=" * 60)
print("Input:\n", CLINICAL_TEXT, "\n")

r = requests.post(
    f"{BASE_URL}/generate/note",
    headers=HEADERS,
    json={"text": CLINICAL_TEXT},
    timeout=240
)

if r.status_code == 200:
    data = r.json()
    REFINED_TEXT = data["text"]
    print("✓ Refined Summary:\n")
    print(REFINED_TEXT)
    print("\nToken Usage:", data.get("tokens_used", {}))
else:
    print(f"✗ Failed ({r.status_code}):", r.text)
    REFINED_TEXT = CLINICAL_TEXT  # fallback
    print("Using original text as fallback.")

Generate Note  (POST /generate/note)
Input:
 Swollen LL, limited mobility, pain and redness over R LL. Possible PE post THR and TKR. 
IV Streptokinase stat
IV NS 500ml run fast
SC Clean 200U stat
Refer to IR for possible embolectomy 

✓ Refined Summary:

The patient presents with a swollen left lower limb, limited mobility, and pain accompanied by redness over the right lower limb. These clinical findings suggest a possible pulmonary embolism following a history of total hip replacement and total knee replacement. Currently, the patient is experiencing acute symptoms of limb swelling and localized inflammation. The diagnosis of suspected pulmonary embolism is being managed with an immediate treatment plan consisting of a stat dose of intravenous Streptokinase, a 500 ml intravenous bolus of Normal Saline to be run fast, and a stat dose of 200 units of subcutaneous Clexane (Enoxaparin). Additionally, the patient is being referred to Interventional Radiology for a possible embolectomy to 

In [8]:
# ── Block 4: Generate Terms (Full Pipeline) ───────────────────────────────────
# Uses REFINED_TEXT from Block 3 (or falls back to CLINICAL_TEXT)
print("=" * 60)
print("Generate Terms  (POST /generate/terms)")
print("=" * 60)
print("Input text (first 200 chars):", REFINED_TEXT[:200], "...\n")

r = requests.post(
    f"{BASE_URL}/generate/terms",
    headers=HEADERS,
    json={"text": REFINED_TEXT},
    timeout=300
)

if r.status_code == 200:
    data = r.json()
    terms = data.get("terms", {})

    print("✓ Extracted SNOMED-CT Terms:\n")
    for category in ["anatomical_sites", "procedures", "symptoms", "medications"]:
        items = terms.get(category, [])
        if items:
            print(f"  [{category.upper()}]")
            for item in items:
                print(f"    • {item['term']}  |  {item['code']}")

    diagnosis = terms.get("diagnosis", {})
    for sub in ["communicable_disease", "non_communicable_disease"]:
        items = diagnosis.get(sub, [])
        if items:
            print(f"  [DIAGNOSIS — {sub.replace('_', ' ').upper()}]")
            for item in items:
                print(f"    • {item['term']}  |  {item['code']}")

    print("\nToken Usage:")
    for step, usage in data.get("tokens_used", {}).items():
        print(f"  {step}: in={usage['input_token']}  out={usage['output_token']}")
else:
    print(f"✗ Failed ({r.status_code}):", r.text)

Generate Terms  (POST /generate/terms)
Input text (first 200 chars): The patient presents with a swollen left lower limb, limited mobility, and pain accompanied by redness over the right lower limb. These clinical findings suggest a possible pulmonary embolism followin ...

✓ Extracted SNOMED-CT Terms:

  [ANATOMICAL_SITES]
    • Lower limb structure  |  61685007
    • Hip joint structure  |  24136001
    • Knee joint structure  |  49076000
  [PROCEDURES]
    • Total replacement of hip  |  52734007
    • Total knee replacement  |  609588000
    • Removal of embolus  |  71815002
    • Administration of intravenous fluid bolus  |  431393006
  [SYMPTOMS]
    • Swelling of limb  |  80068009
    • Pain  |  22253000
    • Erythema  |  247441003
    • Impaired mobility  |  82971005
  [MEDICATIONS]
    • Streptokinase  |  395889004
    • Enoxaparin  |  372562003
  [DIAGNOSIS — NON COMMUNICABLE DISEASE]
    • Pulmonary embolism  |  59282003

Token Usage:
  filter_tags: in=1723  out=395
  valida

In [ ]:
# ── Block 5: MedCAT Health Check ──────────────────────────────────────────────
print("=" * 60)
print("MedCAT Health Check  (GET /generate/medcat/health)")
print("=" * 60)

r = requests.get(
    f"{BASE_URL}/generate/medcat/health",
    headers=HEADERS,
    timeout=120
)

if r.status_code == 200:
    data = r.json()
    alive   = data.get("alive", False)
    status  = data.get("status", "unknown")
    total   = data.get("total_terms", 0)
    error   = data.get("error", None)

    icon = "✓" if alive else "✗"
    print(f"{icon} Status : {status}")
    print(f"  Alive  : {alive}")
    print(f"  Terms  : {total}")
    if error:
        print(f"  Error  : {error}")
else:
    print(f"✗ Failed ({r.status_code}):", r.text)


In [21]:
import requests

def get_mapping(concept_id, mapping_type="icd10"):
    # Map types to RefSet IDs
    refsets = {
        "icd10": "447562003", # Complex map
        "icd9":  "447563008", # Complex map
        "icd11": "1215115004" # ICD-11 MMS Map
    }
    
    target_refset = refsets.get(mapping_type)
    url = f"http://localhost:8080/browser/MAIN/members"
    
    params = {
        'referenceSet': target_refset,
        'referencedComponentId': concept_id,
        'active': 'true'
    }
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        items = response.json().get('items', [])
        
        return [i['additionalFields'].get('mapTarget') for i in items if 'mapTarget' in i['additionalFields']]
    except Exception as e:
        return f"Error: {e}"

# Example usage for ICD-9
snomed_id = "59282003" # Pulmonary embolism
print(f"ICD-9: {get_mapping(snomed_id, 'icd9')}")
print(f"ICD-10: {get_mapping(snomed_id, 'icd10')}")
print(f"ICD-11: {get_mapping(snomed_id, 'icd11')}")

ICD-9: []
ICD-10: ['I26.9']
ICD-11: []


In [22]:
import requests

def list_loaded_refsets():
    url = "http://localhost:8080/fhir/ValueSet/$expand"
    params = {'url': 'http://snomed.info/sct?fhir_vs=refset'}
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        expansion = response.json().get('expansion', {}).get('contains', [])
        
        print(f"{'ID':<18} | {'Display Name'}")
        print("-" * 50)
        for item in expansion:
            # We filter for 'map' to find relevant ones
            if 'map' in item['display'].lower():
                print(f"{item['code']:<18} | {item['display']}")
                
    except Exception as e:
        print(f"Error connecting to Snowstorm: {e}")

list_loaded_refsets()

ID                 | Display Name
--------------------------------------------------
900000000000497000 | CTV3 to SNOMED CT simple map
447562003          | SNOMED CT to ICD-10 extended map
446608001          | SNOMED CT to ICD-O simple map
